# graphArt_LM: per-instance module trees, two GPUs

Fine-tunes a fresh LoRA on `Qwen2.5-Coder-1.5B-Instruct` for one step of the hierarchical
schema, split up: instead of the whole module tree at once, the model writes **one class
instance at a time** -- given the class's code and the arguments its constructor receives,
its child modules, with child instances of classes in the code written as
`name: Class(arg=value, ...)` (each gets its own call later). The root call gets the whole
program. Code joins the answers into the tree. Training data: the same 1,810 training
programs as the hierarchical adapter (generator v3, timm, tiers 1-4), one example per
distinct instance.

Steps: data check -> memory check on the longest examples -> 8-step smoke run on both GPUs
(must show the loss falling) -> one epoch on both GPUs. Expected ~1.8 h, capped at 2.5 h.
No scoring in this notebook.

**Before running**: *GPU T4 x2* (both are used), *Internet on*. No inputs needed. Then
*Save Version -> Save & Run All (Commit)*.

**When it finishes**, download `graphart_hier_inst_train.zip` from the *Output* tab: the
adapter (`lora/qwen2.5-coder-1.5b-graphart-lora-hier-inst/`, with `train_summary.json`) and
the logs. If the time guard stopped training early, the best checkpoint so far is saved.

Generated by `scripts/make_kaggle_notebook.py --hier-inst-train`. Edit that script, not this file.

In [ ]:
# BASE_MODEL   tuned from scratch (a fresh LoRA), not from any earlier adapter.
# ADAPTER      its name; the adapter lands in lora/<ADAPTER>/ inside the zip.
# EPOCHS       one pass over the per-instance examples (12,304 train, 666 val, ~8.6M
#              tokens, ~520 steps of 16).
# GPUS, EFFECTIVE_BATCH  both T4s under torchrun, 8 of each step's 16 examples per GPU.
# TRAIN_ARGS   --hier-inst: one example per distinct class instance of each training program
#              (hier_inst prompt): the root gets the whole reduced program, every other
#              instance its class's code and its constructor arguments; the answer is that
#              instance's children. --hier-inst-max 12: per program the root and 11 instances
#              spread over the rest (of 17,044 in all), to fit the session. Val loss every
#              100 steps picks the checkpoint.
#              --max-len 16384 covers every example (longest 8,204 tokens).
# FALLBACK_MAX_LEN  if the memory check fails, train with this instead; longer examples are
#              left out, and the log names them.
# MEMORY_CHECK forward + backward on this many of the longest examples, before anything else.
# EVAL_RESERVE_HOURS  kept back at the end for saving and zipping (no scoring here).
# TIME_BUDGET_HOURS  the whole session; the time guard stops training early enough to save
#              the best checkpoint so far.
# VALIDATE_ONLY True: only check the code arrived intact (works on a CPU session).
BASE_MODEL = "Qwen/Qwen2.5-Coder-1.5B-Instruct"; ADAPTER = "qwen2.5-coder-1.5b-graphart-lora-hier-inst"; EPOCHS = 1.0; GPUS = 2; EFFECTIVE_BATCH = 16; TRAIN_ARGS = ["--hier-inst", "--hier-inst-max", "12", "--max-len", "16384", "--drop-over-max", "--eval-steps", "100"]; FALLBACK_MAX_LEN = 12288; MEMORY_CHECK = 6; SMOKE_STEPS = 8; EVAL_RESERVE_HOURS = 0.1; TIME_BUDGET_HOURS = 2.5; VALIDATE_ONLY = False

## 1. Unpack the code, prompts and samples

In [ ]:
import base64, hashlib, io, json, os, shutil, subprocess, sys, tarfile, time, urllib.request
from pathlib import Path

START = time.time()
WORK = Path(os.environ.get("GRAPHART_WORK", "/kaggle/working"))
REPO = WORK / "graphArt_LM"
BUNDLE_SHA256 = "2808340a39a4a28dae58ccd8ec2e40a8a7012ae16106adb0a9eea23b8380f054"
BUNDLE = (
    "..."
)
data = base64.b64decode(BUNDLE)
assert hashlib.sha256(data).hexdigest() == BUNDLE_SHA256, "bundle damaged -- regenerate the notebook"
REPO.mkdir(parents=True, exist_ok=True)
with tarfile.open(fileobj=io.BytesIO(data), mode="r:xz") as tar:
    tar.extractall(REPO)
os.chdir(REPO)
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))
print(sorted(str(p.relative_to(REPO)) for p in REPO.rglob("*") if p.is_file()))
if VALIDATE_ONLY:
    raise SystemExit("bundle intact -- VALIDATE_ONLY is set, so stopping here")
subprocess.run("nvidia-smi --query-gpu=name,memory.total --format=csv", shell=True)

## 2. Install the pinned libraries

In [ ]:
from importlib.metadata import PackageNotFoundError, version

PINS = {'transformers': '5.20.2', 'peft': '0.21.2', 'accelerate': '1.15.0'}

def pinned(package, wanted):
    try:
        return version(package) == wanted
    except PackageNotFoundError:
        return False

missing = [f"{p}=={v}" for p, v in PINS.items() if not pinned(p, v)]
if missing:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *missing], check=True)

# peft probes two optional quantisation libraries when it builds LoRA layers, and
# raises -- rather than skipping them -- if they are installed but older than it
# supports. Kaggle ships torchao 0.10 (peft wants >= 0.16), which killed a run. We
# use neither, so remove them if present.
for pkg in ("torchao", "gptqmodel"):
    try:
        print(f"removing {pkg} {version(pkg)} (peft rejects old versions; not used here)", flush=True)
        subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "-q", pkg], check=True)
    except PackageNotFoundError:
        pass

subprocess.run([sys.executable, "-c", "import torch, transformers, peft; "
                "print('torch', torch.__version__, '| cuda', torch.cuda.is_available(), "
                "'| transformers', transformers.__version__, '| peft', peft.__version__)"])

## 3. Download the base model

In [ ]:
# Its own step with a time limit: a stalled download inside training would otherwise
# sit until the session cap. ~3 GB; a few minutes on Kaggle's network.
FETCH = ("from huggingface_hub import snapshot_download; "
         f"print('downloaded to', snapshot_download({BASE_MODEL!r}))")
for attempt in (1, 2):
    try:
        subprocess.run([sys.executable, "-c", FETCH], check=True, timeout=1800)
        break
    except (subprocess.TimeoutExpired, subprocess.CalledProcessError) as exc:
        print(f"download attempt {attempt} failed: {exc}", flush=True)
else:
    raise SystemExit("could not download the base model -- stopping before any training")

## 4. Two GPUs? Then check the data and the GPU memory

In [ ]:
import re, zipfile

n = int(subprocess.run([sys.executable, "-c", "import torch; print(torch.cuda.device_count())"],
                       capture_output=True, text=True).stdout.strip() or 0)
print(f"{n} GPU(s) visible", flush=True)
if n < GPUS:
    raise SystemExit(f"this session has {n} GPU(s), not {GPUS} -- set the accelerator to GPU T4 x2")

ONE_GPU = dict(os.environ, CUDA_VISIBLE_DEVICES="0", HF_HUB_OFFLINE="1", PYTHONUNBUFFERED="1")
ALL_GPUS = dict(ONE_GPU, CUDA_VISIBLE_DEVICES=",".join(str(i) for i in range(GPUS)))
TORCHRUN = [sys.executable, "-m", "torch.distributed.run", "--standalone", "--nproc_per_node", str(GPUS)]
ZIP_NAME = "graphart_hier_inst_train.zip"
LOGS = REPO / "logs"
LOGS.mkdir(exist_ok=True)

def remaining():
    return TIME_BUDGET_HOURS * 3600 - (time.time() - START)

def run(cmd, env):
    # Run, echo the output live, and return (exit code, output).
    proc = subprocess.Popen(cmd, env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    lines = []
    for line in proc.stdout:
        print(line, end="", flush=True)
        lines.append(line)
    return proc.wait(), "".join(lines)

def check(*extra):
    # One process on one GPU: the data check and the memory check.
    return subprocess.run([sys.executable, "scripts/train_lora.py", "--base", BASE_MODEL, *TRAIN_ARGS, *extra],
                          env=ONE_GPU).returncode

def train(*extra):
    # Both GPUs: each step's EFFECTIVE_BATCH examples split between them.
    return run([*TORCHRUN, "scripts/train_lora.py", "--base", BASE_MODEL, *TRAIN_ARGS,
                "--grad-accum", str(EFFECTIVE_BATCH // GPUS), *extra], ALL_GPUS)

def package():
    with zipfile.ZipFile(WORK / ZIP_NAME, "w", zipfile.ZIP_DEFLATED) as z:
        for root in (REPO / "results", REPO / "lora" / ADAPTER, LOGS):
            for p in root.rglob("*") if root.exists() else []:
                if p.is_file() and "checkpoints" not in p.parts:
                    z.write(p, p.relative_to(REPO))

def with_max_len(args, n):
    out = list(args)
    out[out.index("--max-len") + 1] = str(n)
    return out

if check("--out", "lora/check", "--check-only") != 0:
    raise SystemExit("the data check failed -- see the traceback above")
code = check("--out", "lora/check", "--memory-check", str(MEMORY_CHECK))
if code == 5:                                 # train_lora.py: out of memory
    print(f"the longest examples do not fit -- falling back to --max-len {FALLBACK_MAX_LEN}", flush=True)
    TRAIN_ARGS = with_max_len(TRAIN_ARGS, FALLBACK_MAX_LEN)
    code = check("--out", "lora/check", "--memory-check", str(MEMORY_CHECK))
if code != 0:
    raise SystemExit(f"the memory check failed (exit {code}) -- see the log above")

## 5. Smoke run on both GPUs: is it learning?

In [ ]:
# Under torchrun a failing process's own exit code is lost, so the cause is read from the
# log: "NOT LEARNING" or a non-finite loss can be fp16 trouble, worth one retry in fp32.
code, out = train("--out", "lora/smoke", "--smoke-steps", str(SMOKE_STEPS))
if code != 0 and ("NOT LEARNING" in out or "STOPPED: loss is" in out):
    print(f"smoke run failed in fp16 -- retrying in fp32", flush=True)
    code, out = train("--out", "lora/smoke", "--smoke-steps", str(SMOKE_STEPS), "--fp32")
    if code == 0:
        TRAIN_ARGS = [*TRAIN_ARGS, "--fp32"]
if code != 0:
    raise SystemExit(f"the smoke run failed (exit {code}) -- stopping before the full run")

## 6. Train on both GPUs

In [ ]:
hours = max(remaining() / 3600 - EVAL_RESERVE_HOURS, 0.25)
print(f"training for at most {hours:.2f} h", flush=True)
code, out = train("--out", f"lora/{ADAPTER}", "--epochs", str(EPOCHS), "--time-budget-hours", f"{hours:.2f}")
(LOGS / "train.log").write_text(out)
if code != 0:
    package()
    raise SystemExit("training failed -- see the log above")
print((REPO / "lora" / ADAPTER / "train_summary.json").read_text())
package()

## 7. Done

In [ ]:
summary = json.loads((REPO / "lora" / ADAPTER / "train_summary.json").read_text())
print(f"steps {summary['steps']}, {summary['minutes']} min, best val loss {summary['best_eval_loss']}")
for h in summary["eval_history"]:
    print(f"  step {h['step']:4d}  val loss {h['eval_loss']:.4f}")
package()
print(f"done in {(time.time() - START) / 3600:.1f} h -- download {ZIP_NAME}")